## LangGraph

**LangGraph-> is a graph orchestration framework on the top the langchain, where your app is modeled as nodes(functions) and edges(transitions),with shared state flowing through.In real agent  for control flow we need branching, looping and retrying until the result gives success**

#### Part 1: **State-the heart of the langgraph**

In [2]:
from typing import TypedDict, Annotated
from operator import add

class AgentState(TypedDict):
    messages: Annotated[list,add] # add= reducer, it appends the messages not overwrite it
    question: str
    answer: str

#### Part-2: **Node**

In [ ]:
def get_question(state:AgentState)-> AgentState: 
    return {"question":"What is langgraph"}

def answer_question(state:AgentState)-> AgentState:
    ans = f"Answer to :{state['question']}"
    return {"answer":ans}

#node function always return dict with only keys it wants to update - not the full state 

#### Part 3: **Building the state Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("get_q",get_question)
builder.add_node("get_ans",answer_question)

builder.add_edge(START,"get_q")
builder.add_edge("get_q","get_ans") # edges are the transistion state for langgraph
builder.add_edge("get_ans",END)

graph = builder.compile()

result = graph.invoke({"messages":[], "question":"","answer":""})
print(result)

{'messages': [], 'question': 'What is langgraph', 'answer': 'Answer to :What is langgraph'}


#### Part 4: **Conditional Edges**

In [5]:
# Conditional edges are branching the graph besed on the required contions in the graph
# first we have to create the function node with the conditions
def route_decision(state:AgentState):
    if "quantum" in state["question"]:
        return "quantum_node"
    else:
        return "normal_node"

builder.add_conditional_edges(
    "get_q", # source node
    route_decision, # function returning next node name 
    {
        "quantum_node":"quantum_node",
        "normal_node":"normal_node"
    }
    )

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
